In [2]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string

In [3]:
data = pd.read_csv("Symptoms_to_Condition.csv")

In [4]:
data

,symptoms,condition
0,I have muscle spasms and stiffness in lower ba...,Lower back pain
1,I have pain radiating to legs and muscle spasms.,Lower back pain
2,I have difficulty bending and pain in lower back.,Lower back pain
3,I have pain radiating to legs.,Lower back pain
4,I have pain in lower back and muscle spasms an...,Lower back pain
...,...,...
995,I have pain in hip and swelling in hip.,Hip bursitis
996,I have pain in hip.,Hip bursitis
997,I have swelling in hip and pain in hip and pai...,Hip bursitis
998,I have pain when climbing stairs.,Hip bursitis


In [5]:
# Initialize tools
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [6]:
# Preprocessing function
def preprocess_text(text):
    text = text.lower()  # Lowercase
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    tokens = text.split()  # Tokenize
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]  # Lemmatize, remove stop words
    return ' '.join(tokens)

In [7]:
# Apply preprocessing
data['symptoms'] = data['symptoms'].apply(preprocess_text)

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [9]:
# Initialize TF-IDF vectorizer
tfidf = TfidfVectorizer(max_features=1000)

# Fit and transform the symptoms
X = tfidf.fit_transform(data['symptoms']).toarray()
y = data['condition']  # Target labels

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Initialize models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Naive Bayes': MultinomialNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='linear', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

In [12]:
# Train and evaluate each model
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)  # Train
    y_pred = model.predict(X_test)  # Predict
    accuracy = accuracy_score(y_test, y_pred)  # Evaluate
    results[name] = accuracy
    print(f"{name} Accuracy: {accuracy:.4f}")

Logistic Regression Accuracy: 0.9400
Naive Bayes Accuracy: 0.9550
Random Forest Accuracy: 0.9450
SVM Accuracy: 0.9650
Gradient Boosting Accuracy: 0.9450


In [13]:
# New symptom
new_symptom = "I have pain in lower back and stiffness."
new_symptom_processed = preprocess_text(new_symptom)
new_symptom_vec = tfidf.transform([new_symptom_processed]).toarray()

# Predict with SVM
best_model = models['SVM']
predicted_condition = best_model.predict(new_symptom_vec)[0]
print(f"Predicted Condition: {predicted_condition}")

# Load exercise data and recommend
exercise_data = pd.read_csv("Condition_to_Exercises.csv")
exercises = exercise_data[exercise_data['condition'] == predicted_condition]
for index, row in exercises.iterrows():
    print(f"Exercise: {row['exercise']}")
    print(f"Description: {row['description']}\n")

Predicted Condition: Lower back pain
Exercise: Cat-cow stretch
Description: Start on your hands and knees. Arch your back upwards while exhaling, then lower your back and lift your head while inhaling. Repeat 10 times.

Exercise: Child's pose
Description: Kneel on the floor, sit back on your heels, and stretch your arms forward, lowering your chest to the ground. Hold for 30 seconds.

Exercise: Pelvic tilts
Description: Lie on your back with knees bent. Tighten your abdominal muscles and push your lower back into the floor. Hold for 5 seconds, repeat 10 times.

Exercise: Bridges
Description: Lie on your back with knees bent. Lift your hips off the floor, forming a straight line from knees to shoulders. Hold for 5 seconds, repeat 10 times.

Exercise: Knee-to-chest stretch
Description: Lie on your back, pull one knee towards your chest, keeping the other leg straight. Hold for 30 seconds, switch legs.

Exercise: Bird dog
Description: Start on hands and knees. Extend one arm forward and t

In [14]:
import joblib

# Assuming 'best_model' is your trained SVM model and 'tfidf' is your fitted TF-IDF vectorizer
joblib.dump(best_model, 'svm_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

['tfidf_vectorizer.pkl']